In [13]:
# Install all required libraries for the Advanced RAG pipeline
!pip install rank-bm25 sentence-transformers langchain-google-genai langchain-core -q

In [14]:
import os
import time
import getpass
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("All libraries imported successfully.")

All libraries imported successfully.


In [15]:
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

Enter your Google API key: ··········


**Part 1 — Document Corpus**

In [16]:
# Corpus of 20 AI/ML documents covering transformers, training, optimization, and retrieval
# Includes technical jargon (BM25, SGD, MLM) that sparse retrieval handles well
corpus = [
    # Attention & Transformers
    "Attention mechanisms allow models to weigh the importance of different tokens when encoding a sequence.",
    "The Transformer architecture relies entirely on self-attention, replacing recurrent layers with parallel attention heads.",
    "Multi-head attention splits queries, keys, and values into multiple heads, enabling the model to attend to information from different representation subspaces.",
    "Positional encodings are added to token embeddings in Transformers because self-attention is permutation-invariant by default.",
    "Cross-attention in encoder-decoder Transformers allows the decoder to attend to relevant parts of the encoder output.",

    # Neural Network Training
    "Backpropagation computes gradients by applying the chain rule from the loss function back through each layer.",
    "Batch normalization normalizes layer inputs to reduce internal covariate shift and stabilize training.",
    "Dropout randomly zeroes neuron outputs during training, acting as a regularizer to prevent overfitting.",

    # Optimization
    "Stochastic Gradient Descent (SGD) updates weights using gradients computed on small mini-batches of data.",
    "The Adam optimizer combines momentum and RMSProp, adapting learning rates per parameter using first and second moment estimates.",
    "Learning rate scheduling reduces the learning rate over time, helping models converge to better minima.",
    "Gradient clipping caps the norm of gradients during backpropagation to prevent exploding gradients in deep networks.",

    # Embeddings & Retrieval
    "Word2Vec learns dense word embeddings by predicting context words (Skip-gram) or the center word (CBOW).",
    "BERT uses Masked Language Modeling (MLM) and Next Sentence Prediction (NSP) as pre-training objectives.",
    "Cosine similarity measures the angle between two embedding vectors, commonly used in semantic search pipelines.",

    # RAG & Retrieval Systems
    "BM25 is a probabilistic sparse retrieval algorithm that scores documents based on term frequency and inverse document frequency.",
    "Retrieval-Augmented Generation (RAG) grounds LLM responses in retrieved documents, reducing hallucinations.",
    "Dense retrieval encodes queries and documents into vector space and retrieves by nearest-neighbor search.",
    "Reciprocal Rank Fusion (RRF) combines ranked lists from multiple retrievers into a single fused ranking.",
    "Cross-encoders jointly encode query and document pairs, producing more accurate relevance scores than bi-encoders but at higher latency.",
]

print(f"Corpus size: {len(corpus)} documents")

Corpus size: 20 documents


**Part 2 — Hybrid Retriever (BM25 + SBERT + RRF)**

In [17]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        """
        Combines BM25 (sparse) and SBERT (dense) retrieval using RRF fusion.
        k=60 is the RRF constant from the original paper — dampens the impact of top ranks.
        """
        self.corpus = corpus
        self.k = k

        # BM25 works on exact token matches — lowercase ensures case-insensitive matching
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # SBERT encodes semantic meaning — encode all docs once and reuse
        print("Loading SBERT model (all-MiniLM-L6-v2)...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        print("HybridRetriever ready.")

    def _bm25_ranks(self, query: str) -> dict:
        # Score all documents and rank them — rank 1 = most relevant
        scores = self.bm25.get_scores(query.lower().split())
        ranked = np.argsort(scores)[::-1]
        return {doc_id: rank + 1 for rank, doc_id in enumerate(ranked)}

    def _sbert_ranks(self, query: str) -> dict:
        # Encode query and compute cosine similarity against all pre-encoded docs
        query_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        norms = np.linalg.norm(self.doc_embeddings, axis=1) * np.linalg.norm(query_vec)
        scores = self.doc_embeddings @ query_vec / norms
        ranked = np.argsort(scores)[::-1]
        return {doc_id: rank + 1 for rank, doc_id in enumerate(ranked)}

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """
        RRF fuses BM25 and SBERT ranks — a document ranked highly by BOTH
        gets a much higher combined score than one ranked highly by just one.
        Returns top_k docs with both individual ranks visible for analysis.
        """
        bm25_ranks = self._bm25_ranks(query)
        sbert_ranks = self._sbert_ranks(query)

        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            r_bm25 = bm25_ranks[doc_id]
            r_sbert = sbert_ranks[doc_id]
            # RRF formula: higher rank (lower number) = bigger contribution
            rrf_scores[doc_id] = (1 / (self.k + r_bm25)) + (1 / (self.k + r_sbert))

        ranked_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)

        return [
            {
                "doc_id": doc_id,
                "rrf_score": round(rrf_scores[doc_id], 6),
                "bm25_rank": bm25_ranks[doc_id],
                "sbert_rank": sbert_ranks[doc_id],
                "text": self.corpus[doc_id],
            }
            for doc_id in ranked_ids[:top_k]
        ]

# Initialize retriever
retriever = HybridRetriever(corpus)

# Test retrieval
print("\nTest retrieval for 'how does attention work?':")
test_results = retriever.retrieve("how does attention work?", top_k=3)
for r in test_results:
    print(f"  [RRF:{r['rrf_score']}] BM25_rank={r['bm25_rank']} SBERT_rank={r['sbert_rank']}")
    print(f"    → {r['text']}")

Loading SBERT model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever ready.

Test retrieval for 'how does attention work?':
  [RRF:0.032522] BM25_rank=2 SBERT_rank=1
    → Attention mechanisms allow models to weigh the importance of different tokens when encoding a sequence.
  [RRF:0.032266] BM25_rank=1 SBERT_rank=3
    → The Transformer architecture relies entirely on self-attention, replacing recurrent layers with parallel attention heads.
  [RRF:0.032002] BM25_rank=3 SBERT_rank=2
    → Multi-head attention splits queries, keys, and values into multiple heads, enabling the model to attend to information from different representation subspaces.


**Part 3 — Cross-Encoder Re-Ranker**

In [18]:
# Cross-encoder jointly reads query + document together for precise relevance scoring
# Much more accurate than bi-encoders but slower — used only on top candidates
print("Loading Cross-Encoder (ms-marco-MiniLM-L-6-v2)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-Encoder ready.")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-ranks retrieved candidates using cross-encoder scores.
    Always uses the ORIGINAL user query — not the HyDE-expanded version —
    because we want relevance to what the user actually asked.
    Cross-encoder scores can be negative — higher (less negative) = more relevant.
    """
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, score in enumerate(scores):
        candidates[i]["cross_encoder_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked[:top_k]

# Test re-ranker
print("\nTest re-ranking for 'how does attention work?':")
reranked = rerank("how does attention work?", test_results)
for r in reranked:
    print(f"  [CE score: {r['cross_encoder_score']}] {r['text']}")

Loading Cross-Encoder (ms-marco-MiniLM-L-6-v2)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder ready.

Test re-ranking for 'how does attention work?':
  [CE score: 5.287] Attention mechanisms allow models to weigh the importance of different tokens when encoding a sequence.
  [CE score: 0.2717] Multi-head attention splits queries, keys, and values into multiple heads, enabling the model to attend to information from different representation subspaces.
  [CE score: -3.554] The Transformer architecture relies entirely on self-attention, replacing recurrent layers with parallel attention heads.


**Part 4 — Query Expansion (HyDE)**

In [19]:
# Initialize Gemini LLM — temperature=0.0 for deterministic, factual outputs
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# HyDE prompt: make the LLM write a textbook-style answer to bridge the vocabulary gap
# between short student queries and technical document language
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Generate a single factual paragraph (3-5 sentences) "
               "that directly answers the question. Write it as if it were an excerpt from an AI/ML textbook."),
    ("human", "{query}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

def hyde_expand(query: str, retries: int = 5, wait: int = 90) -> str:
    """
    HyDE (Hypothetical Document Embeddings): instead of embedding the raw short query,
    we generate a full hypothetical answer and embed that.
    This bridges the vocabulary gap between vague student queries and precise documents.
    Falls back to the original query if all retries fail.
    """
    for i in range(retries):
        try:
            return hyde_chain.invoke({"query": query})
        except Exception as e:
            if "429" in str(e) and i < retries - 1:
                print(f"Rate limited. Attempt {i+1}/{retries}. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"HyDE failed after {retries} attempts. Using original query.")
                return query  # graceful fallback

# Test HyDE
query = "how does attention work?"
hyp_doc = hyde_expand(query)
print("Original Query:", query)
print("\nHyDE Hypothetical Document:")
print(hyp_doc)

Original Query: how does attention work?

HyDE Hypothetical Document:
Attention mechanisms allow a neural network to dynamically weigh the importance of different parts of an input sequence when processing a specific element or generating an output. This process typically involves computing a similarity score between a "query" vector, representing the current focus, and all "key" vectors, which represent potential information sources within the input. These raw scores are then normalized, commonly using a softmax function, to produce a set of attention weights that sum to one. Finally, these weights are applied to corresponding "value" vectors, creating a weighted sum that serves as a context-aware representation for the query. This selective focusing enables models to capture long-range dependencies and prioritize relevant information, significantly enhancing performance in tasks like machine translation and natural language understanding.


**Part 5 — End-to-End Pipeline**

In [20]:
def advanced_rag(user_query: str) -> str:
    """
    Full Advanced RAG pipeline:
    1. HyDE expands the vague query into a rich hypothetical document
    2. Hybrid retrieval (BM25 + SBERT + RRF) retrieves top 8 candidates
    3. Cross-encoder re-ranks using the ORIGINAL query for precision
    4. Gemini generates a grounded answer from top 3 documents
    """
    print(f"\n{'='*60}")
    print(f"Query: {user_query}")
    print('='*60)

    # Step 1: HyDE expansion — richer text = better retrieval signal
    print("\n[1/4] Generating HyDE expansion...")
    expanded_query = hyde_expand(user_query)
    print(f"  HyDE doc (truncated): {expanded_query[:150]}...")

    # Step 2: Hybrid retrieval on the expanded query
    print("\n[2/4] Hybrid Retrieval (BM25 + SBERT + RRF)...")
    candidates = retriever.retrieve(expanded_query, top_k=8)
    print(f"  Retrieved {len(candidates)} candidates:")
    for c in candidates:
        print(f"    [RRF:{c['rrf_score']}] BM25={c['bm25_rank']} SBERT={c['sbert_rank']} → {c['text'][:65]}...")

    # Step 3: Re-rank with cross-encoder on original query
    print("\n[3/4] Cross-Encoder Re-Ranking...")
    top_docs = rerank(user_query, candidates, top_k=3)
    print("  Top 3 after re-ranking:")
    for d in top_docs:
        print(f"    [CE:{d['cross_encoder_score']}] {d['text'][:70]}...")

    # Step 4: Generate final answer grounded in retrieved context
    print("\n[4/4] Generating Answer with Gemini...")
    context = "\n".join([f"- {d['text']}" for d in top_docs])

    answer_prompt = f"""You are a helpful AI/ML tutor. Answer the student's question using ONLY the context below.
Be concise and accurate.

Context:
{context}

Question: {user_query}
Answer:"""

    response = llm.invoke(answer_prompt)
    answer = response.content
    print(f"\nFinal Answer:\n{answer}")
    return answer

# Test the full pipeline
answer = advanced_rag("how do transformers encode meaning?")


Query: how do transformers encode meaning?

[1/4] Generating HyDE expansion...
  HyDE doc (truncated): Transformers encode meaning primarily through their self-attention mechanism, which allows each word in an input sequence to weigh the importance of e...

[2/4] Hybrid Retrieval (BM25 + SBERT + RRF)...
  Retrieved 8 candidates:
    [RRF:0.032787] BM25=1 SBERT=1 → Positional encodings are added to token embeddings in Transformer...
    [RRF:0.032258] BM25=2 SBERT=2 → Attention mechanisms allow models to weigh the importance of diff...
    [RRF:0.031258] BM25=3 SBERT=5 → Multi-head attention splits queries, keys, and values into multip...
    [RRF:0.03125] BM25=4 SBERT=4 → Cross-attention in encoder-decoder Transformers allows the decode...
    [RRF:0.029857] BM25=6 SBERT=8 → Word2Vec learns dense word embeddings by predicting context words...
    [RRF:0.029199] BM25=8 SBERT=9 → Cross-encoders jointly encode query and document pairs, producing...
    [RRF:0.02901] BM25=11 SBERT=7 → Cos

**Part 6 — Naive RAG vs Advanced RAG Comparison**

In [23]:
# Naïve RAG: simplest possible retrieval — just SBERT cosine similarity on the raw query.
# No query expansion, no BM25, no re-ranking. Gets the semantically closest doc but
# misses keyword-specific matches and cannot handle vocabulary mismatch.
def naive_rag(user_query: str) -> str:
    query_vec = retriever.sbert.encode([user_query], convert_to_numpy=True)[0]
    norms = np.linalg.norm(retriever.doc_embeddings, axis=1) * np.linalg.norm(query_vec)
    scores = retriever.doc_embeddings @ query_vec / norms
    top_id = int(np.argmax(scores))
    return corpus[top_id]

def advanced_rag_top_doc(user_query: str) -> str:
    """Returns only the top doc from the advanced pipeline (for comparison)."""
    expanded = hyde_expand(user_query)
    time.sleep(3)
    candidates = retriever.retrieve(expanded, top_k=8)
    top_docs = rerank(user_query, candidates, top_k=1)
    return top_docs[0]["text"]

# 3 test queries — last one is your own
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is the difference between dense and sparse retrieval?",
]

rows = []
print("Running Naïve RAG vs Advanced RAG comparison...\n")

for q in test_queries:
    print(f"▶ Query: {q}")
    naive_top = naive_rag(q)
    print(f"  Naïve RAG  → {naive_top[:80]}...")
    time.sleep(10)  # avoid rate limiting between LLM calls
    adv_top = advanced_rag_top_doc(q)
    print(f"  Advanced   → {adv_top[:80]}...")
    different = " Yes" if naive_top != adv_top else " No"
    print(f"  Different? {different}\n")
    rows.append((q, naive_top, adv_top, different))

# Print final markdown table
print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)
print("| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |")
print("|---|---|---|---|")
for q, naive_top, adv_top, diff in rows:
    print(f"| {q} | {naive_top[:55]}... | {adv_top[:55]}... | {diff} |")

Running Naïve RAG vs Advanced RAG comparison...

▶ Query: how do transformers encode meaning?
  Naïve RAG  → Positional encodings are added to token embeddings in Transformers because self-...
  Advanced   → Positional encodings are added to token embeddings in Transformers because self-...
  Different?  No

▶ Query: optimization techniques for training
  Naïve RAG  → The Adam optimizer combines momentum and RMSProp, adapting learning rates per pa...
  Advanced   → BERT uses Masked Language Modeling (MLM) and Next Sentence Prediction (NSP) as p...
  Different?  Yes

▶ Query: what is the difference between dense and sparse retrieval?
  Naïve RAG  → Dense retrieval encodes queries and documents into vector space and retrieves by...
  Advanced   → Dense retrieval encodes queries and documents into vector space and retrieves by...
  Different?  No


COMPARISON TABLE
| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| how do transformers encode meaning? | P

## Observations

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | Positional encodings are added to token embeddings in Transf... | Positional encodings are added to token embeddings in Transf... | No |
| optimization techniques for training | The Adam optimizer combines momentum and RMSProp, adapting l... |  BERT uses Masked Language Modeling (MLM) and Next Sentence P...  | Yes |
| what is the difference between dense and sparse retrieval? | Dense retrieval encodes queries and documents into vector sp... |  Dense retrieval encodes queries and documents into vector sp... | No |

### Key Observations:
- HyDE expansion enriches short vague queries with technical vocabulary,
  helping BM25 find keyword matches it would otherwise miss.
- The cross-encoder re-ranker reorders results using the original query,
  correcting any drift introduced by the hypothetical document.
- Hybrid retrieval (BM25 + SBERT + RRF) outperforms dense-only retrieval
  on keyword-heavy queries like "BM25" or "SGD optimizer".